## 加载模型和编码器

In [1]:
import numpy as np

In [2]:
def load_model_and_encoders():
    """
    加载所有模型、编码器和标准化器。
    
    返回：
    - model: XGBoost模型
    - label_encoders: 分类特征的标签编码器字典
    - scaler_x: 特征标准化器
    - scaler_y: 目标值标准化器
    - city_map: 城市频率编码映射
    - city_labels: 城市标签编码映射
    - city_embeddings: 城市嵌入编码字典
    """
    import xgboost as xgb
    import joblib
    import json
    import pandas as pd
    
    # 1. 加载XGBoost模型
    model = xgb.XGBRegressor()
    model.load_model("../../data-hh/my/模型文件/频率编码/归一化_xgboost_model_1000.json")
    
    # 2. 加载标签编码器
    categorical_columns = ['flt_no', 'bd_type', 'aircraft']
    label_encoders = {}
    for col in categorical_columns:
        label_encoders[col] = joblib.load(f"../../data-hh/my/encoder/{col}_encoder_all.pkl")
    
    # 3. 加载标准化器
    scaler_x = joblib.load('../../data-hh/my/encoder/standard_scaler_x.pkl')
    scaler_y = joblib.load('../../data-hh/my/encoder/standard_scaler_y.pkl')
    
    # 4. 加载城市频率编码映射
    with open('../../data-hh/my/encoder/city_map_频率编码.json', 'r') as f:
        city_map = json.load(f)
    
    # 5. 加载城市标签编码映射
    with open('../../data-hh/my/encoder/city_labels_航班频率加权图标签.json', 'r') as f:
        city_labels = json.load(f)
    
    # 6. 加载城市嵌入编码（修改了文件路径）
    with open('../../data-hh/my/encoder/城市嵌入编码_航班频率加权图.json', 'r') as f:
        city_embeddings = json.load(f)
    
    print("所有模型和编码器加载完成")
    
    return model, label_encoders, scaler_x, scaler_y, city_map, city_labels, city_embeddings


## 数据预处理

In [3]:
def preprocess_data(flt_date, flt_no, bd_type, dep_time, ab_cap, bc_cap, ac_cap, aircraft, 
                   ab_duration, bc_duration, ac_duration, 
                   a, b, c, ab_price, bc_price, ac_price, 
                   label_encoders, scaler_x, city_map, city_labels, city_embeddings):
    """
    对输入数据进行预处理，生成模型输入。
    """
    import pandas as pd
    import numpy as np
    from datetime import datetime
    
    # 创建三条记录的DataFrame（对应三个leg_no）
    data = []
    
    # 处理日期和时间
    flt_datetime = datetime.strptime(f"{flt_date} {dep_time}", "%Y-%m-%d %H:%M:%S")
    year = flt_datetime.year
    month = flt_datetime.month
    day = flt_datetime.day
    weekday = flt_datetime.weekday()
    hour = flt_datetime.hour
    minute = flt_datetime.minute
    second = flt_datetime.second
    
    # 处理unit_price, duration和cap
    leg_info = {
        1: {'price': ab_price, 'duration': ab_duration, 'from': a, 'to': b, 'cap': ab_cap},
        2: {'price': bc_price, 'duration': bc_duration, 'from': b, 'to': c, 'cap': bc_cap},
        3: {'price': ac_price, 'duration': ac_duration, 'from': a, 'to': c, 'cap': ac_cap}
    }
    
    # 为每个leg_no创建一条记录
    for leg_no in [1, 2, 3]:
        record = {
            # 基础特征编码
            'flt_no': label_encoders['flt_no'].transform([flt_no])[0],
            'bd_type': label_encoders['bd_type'].transform([bd_type])[0],
            'cap': float(leg_info[leg_no]['cap']) if not pd.isna(leg_info[leg_no]['cap']) else 0.0,
            'aircraft': label_encoders['aircraft'].transform([aircraft])[0],
            'legs': 3,
            'leg_no': leg_no,
            'duration': float(leg_info[leg_no]['duration']) if not pd.isna(leg_info[leg_no]['duration']) else 0.0,
            
            # 城市频率编码
            'a': city_map.get(a, 0),
            'b': city_map.get(b, 0),
            'c': city_map.get(c, 0),
            'from': city_map.get(leg_info[leg_no]['from'], 0),
            'to': city_map.get(leg_info[leg_no]['to'], 0),
            
            # 时间特征
            'year': year,
            'month': month,
            'day': day,
            'weekday': weekday,
            'hour': hour,
            'minute': minute,
            'second': second,
            
            # 城市标签
            'a_label': city_labels.get(a, -1),
            'b_label': city_labels.get(b, -1),
            'c_label': city_labels.get(c, -1),
            'from_label': city_labels.get(leg_info[leg_no]['from'], -1),
            'to_label': city_labels.get(leg_info[leg_no]['to'], -1),
            
            # 票价
            'unit_price': float(leg_info[leg_no]['price']) if not pd.isna(leg_info[leg_no]['price']) else 0.0
        }
        
        # 添加城市嵌入编码
        city_values = {
            'a': a,
            'b': b,
            'c': c,
            'from': leg_info[leg_no]['from'],
            'to': leg_info[leg_no]['to']
        }
        
        for prefix, city_id in city_values.items():
            if city_id in city_embeddings:
                record[f'{prefix}_embedding_1'] = city_embeddings[city_id][0]
                record[f'{prefix}_embedding_2'] = city_embeddings[city_id][1]
            else:
                record[f'{prefix}_embedding_1'] = 0.0
                record[f'{prefix}_embedding_2'] = 0.0
        
        data.append(record)
    
    # 转换为DataFrame
    df = pd.DataFrame(data)
    
    # 确保列的顺序与训练时一致
    expected_columns = ['flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 
                       'a', 'b', 'c', 'year', 'month', 'day', 'weekday', 'hour', 'minute', 'second', 
                       'from', 'to', 'unit_price', 'a_label', 'b_label', 'c_label', 'from_label', 'to_label',
                       'a_embedding_1', 'a_embedding_2', 'b_embedding_1', 'b_embedding_2', 
                       'c_embedding_1', 'c_embedding_2', 'from_embedding_1', 'from_embedding_2', 
                       'to_embedding_1', 'to_embedding_2']
    df = df[expected_columns]
    
    # 使用scaler进行标准化
    scaled_data = scaler_x.transform(df)
    scaled_df = pd.DataFrame(scaled_data, columns=df.columns)
    
    return scaled_df

## 客流预测

In [4]:
def predict_pax(processed_data, model, scaler_y):
    """
    使用模型进行客流预测，并反向标准化。
    
    参数:
    - processed_data: DataFrame，经过预处理的数据
    - model: XGBoost模型
    - scaler_y: 目标值的标准化器
    
    返回:
    - dict: 包含三个航段的预测客流量，格式为 {'AB_PAX': xx, 'BC_PAX': xx, 'AC_PAX': xx}
    """
    # 使用模型进行预测（得到标准化的预测值）
    scaled_predictions = model.predict(processed_data)
    
    # 将预测值转换为2D数组（为了使用scaler_y）
    scaled_predictions_2d = scaled_predictions.reshape(-1, 1)
    
    # 使用scaler_y进行反向转换，得到实际的预测值
    predictions = scaler_y.inverse_transform(scaled_predictions_2d)
    
    # 将预测结果映射到对应的航段
    # leg_no=1 对应AB，leg_no=2 对应BC，leg_no=3 对应AC
    result = {
        'AB_PAX': max(0, round(float(predictions[0]))),
        'BC_PAX': max(0, round(float(predictions[1]))),
        'AC_PAX': max(0, round(float(predictions[2])))
    }
    
    return result

## 座位分配

In [5]:
def allocate_seats(pax_predictions, ab_price, bc_price, ac_price, cap):
    """
    根据收益最大化策略，分配航段 AB, BC, AC 的客舱容量。
    
    输入:
    - pax_predictions: 字典，包含 AB, BC, AC 的乘客数预测值，例如 {'AB_PAX': 120, 'BC_PAX': 110, 'AC_PAX': 130}。
    - ab_price: float，航段 AB 的票价。
    - bc_price: float，航段 BC 的票价。
    - ac_price: float，航段 AC 的票价。
    - cap: int，总客舱容量。

    输出:
    - tuple: (allocation, revenue)
        - allocation: 字典，包含分配给 AB, BC, AC 的舱位数量
        - revenue: float，预计总收入
    """
    # 获取航段预测乘客数
    ab_pax = round(pax_predictions['AB_PAX'])
    bc_pax = round(pax_predictions['BC_PAX'])
    ac_pax = round(pax_predictions['AC_PAX'])

    # 短途类（S 类）的最大容量
    s_max = max(ab_pax, bc_pax)
    s_min = min(ab_pax, bc_pax)

    # 分配结果初始化
    allocation = {'AB': 0, 'BC': 0, 'AC': 0}

    # 判断是否满座
    if s_max + ac_pax <= cap:
        # 未满座：按比例分配 S 类和 L 类
        total_pax = s_max + ac_pax
        s_cap = round(cap * (s_max / total_pax))  # S 类分配的容量
        l_cap = cap - s_cap  # L 类分配的容量

        # S 类：直接分配 s_cap（AB 和 BC 同时销售）
        allocation['AB'] = s_cap
        allocation['BC'] = s_cap

        # L 类：分配剩余容量
        allocation['AC'] = l_cap
    else:
        # 已满座：根据收益优先级分配
        s_revenue = ab_price + bc_price
        l_revenue = ac_price

        if s_revenue > l_revenue:
            # 优先分配给 S 类的 min(AB_PAX, BC_PAX)
            s_cap = min(s_min, cap)
            allocation['AB'] = s_cap
            allocation['BC'] = s_cap

            # 剩余容量优先分配给 L 类
            remaining_cap = cap - s_cap
            allocation['AC'] = min(ac_pax, remaining_cap)

            # 如果 L 类分配后还有剩余容量,直接分配给 S 类
            remaining_after_l = remaining_cap - allocation['AC']
            if remaining_after_l > 0:
                allocation['AB'] += remaining_after_l
                allocation['BC'] += remaining_after_l
        else:
            # 优先分配给 L 类
            allocation['AC'] = min(ac_pax, cap)  # L 类尽量满足 AC 的预测乘客数
            remaining_cap = cap - allocation['AC']  # 剩余容量分配给 S 类
            allocation['AB'] = remaining_cap
            allocation['BC'] = remaining_cap

    # 计算预期收入
    revenue = (min(ab_pax, allocation['AB']) * ab_price + 
              min(bc_pax, allocation['BC']) * bc_price + 
              min(ac_pax, allocation['AC']) * ac_price)

    return allocation, revenue


## 测试数据

In [6]:
# 测试数据
test_data = {
    'flt_date': '2024-1-1',
    'flt_no': '/O2MVlG1lP0=',
    'bd_type': '窄体',
    'dep_time': '07:00:00',
    'ab_cap': 194.0,  # AB航段容量
    'bc_cap': 194.0,  # BC航段容量
    'ac_cap': np.nan,  # AC航段容量
    'aircraft': '321',
    'ab_duration': 2.38,    # AB航段时长
    'bc_duration': 2.68,    # BC航段时长
    'ac_duration': np.nan,  # AC航段时长
    'a': 'xIqe+QQIQXQ=',
    'b': 'AfKac4FzdXs=',
    'c': 'So/c8CkA/Xs=',
    'ab_price': 982.02,
    'bc_price': 476.36,
    'ac_price': 1441.70
}

In [7]:
# 测试数据
test_data = {
    'flt_date': '2023-1-1',
    'flt_no': 'P6p42wyZFEI=',  # 从图片中第一行获取
    'bd_type': '窄体',
    'dep_time': '08:50:00',  # 从图片中第一行获取
    'ab_cap': 132.0,  # 从图片中获取
    'bc_cap': 132.0,  # 从图片中获取
    'ac_cap': np.nan,  # 从图片中获取
    'aircraft': '319',  # 从图片中获取
    'ab_duration': 2.38,    # 从图片中第三行leg_no=1的duration获取
    'bc_duration': 2.68,    # 从图片中第二行leg_no=2的duration获取
    'ac_duration': 0.0,     # 从图片中第一行leg_no=3的duration获取
    'a': 'tKjndGSl9NQ=',      # 从图片中航班号的第一部分获取
    'b': 'KETr2NAmAHE=',   # 从图片中航班号的中间部分获取
    'c': '5t+HPO9Mu/w=',      # 从图片中航班号的最后部分获取
    'ab_price': 1048,    # 这些价格可以保持不变
    'bc_price': 1259,    # 因为图片中没有显示价格信息
    'ac_price': 1682
    # 'ab_price': 1048,    # 这些价格可以保持不变
    # 'bc_price': 2000,    # 因为图片中没有显示价格信息
    # 'ac_price': 2000
}

In [8]:
# 加载所有模型和编码器
model, label_encoders, scaler_x, scaler_y, city_map, city_labels, city_embeddings = load_model_and_encoders()

所有模型和编码器加载完成


In [9]:
import numpy as np

# 直接传递所有参数，而不是使用字典解包
processed_data = preprocess_data(
    flt_date=test_data['flt_date'],
    flt_no=test_data['flt_no'],
    bd_type=test_data['bd_type'],
    dep_time=test_data['dep_time'],
    ab_cap=test_data['ab_cap'],
    bc_cap=test_data['bc_cap'],
    ac_cap=test_data['ac_cap'],
    aircraft=test_data['aircraft'],
    ab_duration=test_data['ab_duration'],
    bc_duration=test_data['bc_duration'],
    ac_duration=test_data['ac_duration'],
    a=test_data['a'],
    b=test_data['b'],
    c=test_data['c'],
    ab_price=test_data['ab_price'],
    bc_price=test_data['bc_price'],
    ac_price=test_data['ac_price'],
    label_encoders=label_encoders,
    scaler_x=scaler_x,
    city_map=city_map,
    city_labels=city_labels,
    city_embeddings=city_embeddings
)

In [10]:
# 进行预测
predictions = predict_pax(processed_data, model, scaler_y)
print("预测结果:", predictions)


预测结果: {'AB_PAX': 116, 'BC_PAX': 61, 'AC_PAX': 19}


In [11]:
# 从test_data中选择不为0或nan的cap值
caps = [
    test_data['ab_cap'],
    test_data['bc_cap'],
    test_data['ac_cap']
]
cap = next(c for c in caps if c == c and c != 0)  # 获取第一个不为nan且不为0的值
print(cap)
# 调用座位分配函数



132.0


In [13]:
allocation, revenue = allocate_seats(
    pax_predictions=predictions,
    ab_price=test_data['ab_price'],
    bc_price=test_data['bc_price'],
    ac_price=test_data['ac_price'],
    cap=cap
)

# 打印分配结果
print("座位分配结果:")
print(f"AB航段分配座位: {allocation['AB']}")
print(f"BC航段分配座位: {allocation['BC']}")
print(f"AC航段分配座位: {allocation['AC']}")
print(f"预计总收入: {revenue}")

座位分配结果:
AB航段分配座位: 113.0
BC航段分配座位: 113.0
AC航段分配座位: 19
预计总收入: 227181.0


In [14]:
#     'ab_price': 1048,    # 这些价格可以保持不变
#     'bc_price': 1259,    # 因为图片中没有显示价格信息
#     'ac_price': 1682

# 预测结果: {'AB_PAX': 116, 'BC_PAX': 61, 'AC_PAX': 19}
1048*113+1259*61+19*1682

227181

## 票价预测